In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Qwen2.5-VL zero-shot

Run zero-shot inference and evaluate per-question metrics.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch

from transformers import AutoProcessor, AutoModelForCausalLM
try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None


/workspace/vqa-rag/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    load_label_maps,
    add_label_ids,
    compute_metrics_per_question,
    compute_binary_metrics,
    save_metrics,
    save_predictions,
    normalize_answer,
)

DATA_PATH = find_long_table(ROOT)
LABEL_MAP_DIR = ROOT / "0_dataset_prep" / "out" / "label_maps"
OUT_DIR = ROOT / "3_modern_vlm" / "out" / "05_qwen2_5_vl_zeroshot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = ROOT / "3_modern_vlm" / "results" / "05_qwen2_5_vl_zeroshot"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
BATCH_SIZE = 16
MAX_SAMPLES_PER_SPLIT = int(os.environ.get("MAX_SAMPLES_PER_SPLIT", "0")) or None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_4BIT = False
USE_8BIT = False
GEN_KWARGS = {"max_new_tokens": 16, "do_sample": False}

AUTO_TUNE = True
MIN_GPU_MEM_GB = 24


def _auto_tune():
    global BATCH_SIZE, USE_4BIT, USE_8BIT
    if not AUTO_TUNE or not torch.cuda.is_available():
        return
    try:
        mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    except Exception:
        return
    if mem_gb < MIN_GPU_MEM_GB:
        if BitsAndBytesConfig is not None:
            USE_4BIT = True
            USE_8BIT = False
        BATCH_SIZE = min(BATCH_SIZE, 1)
        print(f"Auto-tune: GPU {mem_gb:.1f}GB -> BATCH_SIZE={BATCH_SIZE}, USE_4BIT={USE_4BIT}")


_auto_tune()


In [4]:
label_maps = load_label_maps(LABEL_MAP_DIR)
long_df = load_long_table(DATA_PATH)

if MAX_SAMPLES_PER_SPLIT:
    long_df = long_df.groupby("split", group_keys=False).head(MAX_SAMPLES_PER_SPLIT)


In [5]:
if (USE_4BIT or USE_8BIT) and BitsAndBytesConfig is None:
    print("Warning: bitsandbytes not installed; disabling 4/8-bit quantization.")
    USE_4BIT = False
    USE_8BIT = False

quant_config = None
if torch.cuda.is_available() and BitsAndBytesConfig is not None:
    if USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
    elif USE_8BIT:
        quant_config = BitsAndBytesConfig(load_in_8bit=True)

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

if AutoModelForVision2Seq is not None:
    try:
        model = AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME,
            device_map="auto" if DEVICE == "cuda" else None,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
            quantization_config=quant_config,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto" if DEVICE == "cuda" else None,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
            quantization_config=quant_config,
        )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto" if DEVICE == "cuda" else None,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
        quantization_config=quant_config,
    )

model.eval()


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLU()

In [6]:
# Prompt formatting

def format_prompt(question: str) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    return f"""USER: <image>\nQuestion: {q}\nASSISTANT:"""


def postprocess(text: str) -> str:
    cleaned = text
    if "ASSISTANT:" in cleaned:
        cleaned = cleaned.split("ASSISTANT:")[-1]
    return cleaned.strip()


In [7]:
# Generate per split
pred_frames = []

for split in sorted(long_df["split"].unique()):
    df_split = long_df[long_df["split"] == split].reset_index(drop=True)
    preds = []
    for start in tqdm(range(0, len(df_split), BATCH_SIZE), desc=f"{split} gen"):
        batch = df_split.iloc[start : start + BATCH_SIZE]
        images = [Image.open(p).convert("RGB") for p in batch["image_path"].tolist()]
        prompts = [format_prompt(q) for q in batch["question_text"].tolist()]
        inputs = processor(images=images, text=prompts, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, **GEN_KWARGS)
        decoded = processor.batch_decode(out, skip_special_tokens=True)
        preds.extend([postprocess(t) for t in decoded])

    out_df = df_split.copy()
    out_df["pred_answer"] = [normalize_answer(p) for p in preds]
    pred_frames.append(out_df)

pred_df = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()


train gen:   0%|          | 0/1835 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature'

validation gen:   0%|          | 0/459 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature'

In [8]:
if len(pred_df):
    pred_df = add_label_ids(pred_df, label_maps, ans_col="answer_norm", out_col="label_id")
    pred_df = add_label_ids(pred_df, label_maps, ans_col="pred_answer", out_col="pred_label_id")

    for split in sorted(pred_df["split"].unique()):
        df_split = pred_df[pred_df["split"] == split]
        overall, per_q = compute_metrics_per_question(df_split, "label_id", "pred_label_id")
        binary = compute_binary_metrics(df_split, label_maps, "label_id", "pred_label_id")
        split_out = RESULTS_DIR / split
        save_metrics(split_out, overall, per_q, binary)
        save_predictions(
            df_split,
            split_out,
            columns=["image_id", "question_id", "answer_norm", "pred_answer", "split"],
        )

OUT_DIR
RESULTS_DIR


PosixPath('/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/3_modern_vlm/results/05_qwen2_5_vl_zeroshot')